# MultiLexNorm2026 Gemma Colab Workflow

This notebook keeps only the cells needed right now: setup, authentication, Gemma LoRA fine-tuning, inspection, validation evaluation, and MFR comparison.

Important split rule: train on `train`, evaluate/tune on `validation`, and reserve `test` for final prediction/submission. Do not train on `test` if you want a meaningful score.


In [ ]:
!nvidia-smi

In [ ]:
%cd /content
import os
repo = "/content/MultiLexNorm2026"
if not os.path.isdir(repo + "/.git"):
    !git clone https://github.com/qwfjop/MultiLexNorm2026.git
else:
    %cd /content/MultiLexNorm2026
    !git pull
%cd /content/MultiLexNorm2026
!git log -1 --oneline

In [ ]:
!pip install -r requirements.txt

## Hugging Face Login

Use a Hugging Face token with gated/public model read access. If you use Colab Secrets, create a secret named `HF_TOKEN`, then run the first cell below. Otherwise run `!hf auth login` and paste the token manually.


In [ ]:
import os
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    !hf auth login --token "$HF_TOKEN"
else:
    print("No HF_TOKEN Colab secret found. Run the next line manually if needed:")
    print("!hf auth login")

In [ ]:
from datasets import load_dataset

data = load_dataset("weerayut/multilexnorm2026-dev-pub", token=True)
print(data)
print("validation languages:", sorted(set(data["validation"]["lang"])))
print("one train row:", data["train"][0])

## Baseline Reference

This is the exact MFR baseline. Our Gemma-backed model must beat this validation ERR, not just lower training loss.


In [ ]:
!python mfr_baseline.py \
  --eval-only \
  --use-auth-token

## Gemma Prompt Contract

`gemma_model.py` trains Gemma with this target behavior:

- Input: language, whole tokenized sentence, target token index, target token.
- Output: exactly one normalized token.
- No JSON, labels, explanations, quotes, markdown, or extra tokens.
- If the token is already normalized, copy it unchanged.
- Preserve punctuation, URLs, mentions, hashtags, emoticons, and placeholders unless the gold labels normalize them.

This strict format matters because evaluation compares token lists exactly.


In [ ]:
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive')
run_id = datetime.now().strftime('%y%m%d_%H%M')
GEMMA_DIR = f'/content/drive/MyDrive/MultiLexNorm2026/models/gemma_colab_{run_id}'
print('GEMMA_DIR=', GEMMA_DIR)

## Train Gemma LoRA

Start with the 270M instruction Gemma because it is practical on Colab. This trains LoRA adapter weights, not a full model copy. Increase `--max-train-examples` after a smoke run succeeds.

Expected output during training:

```text
CUDA
trainable params: ...
training examples=... batches_per_epoch=... batch_size=... grad_accum=... device=cuda lora=True
epoch=1/1 batch=... optimizer_steps=... batch_loss=... avg_loss=...
Saved training metadata to .../training_args.json and .../training_args.txt
```


In [ ]:
!python gemma_model.py \
  --train \
  --use-auth-token \
  --model-name google/gemma-3-270m-it \
  --model-dir "$GEMMA_DIR" \
  --max-train-examples 40000 \
  --epochs 1 \
  --batch-size 4 \
  --gradient-accumulation-steps 8 \
  --learning-rate 2e-4 \
  --max-length 256 \
  --lora-rank 16 \
  --lora-alpha 32 \
  --log-every-steps 50 \
  --save-every-steps 250

## Inspect English and Korean Predictions

Use this before a full evaluation. If predictions include explanations or repeated text, stop and adjust training before spending time on full validation.


In [ ]:
!python gemma_model.py \
  --inspect \
  --use-auth-token \
  --model-dir "$GEMMA_DIR" \
  --batch-size 16 \
  --inspect-limit 10 \
  --inspect-langs en,ko

## Evaluate Standalone Gemma

This evaluates Gemma on the validation split. Start with a subset, then remove `--max-eval-examples` for the full validation run. Standalone Gemma may underperform because it touches every token.


In [ ]:
!python gemma_model.py \
  --eval-only \
  --use-auth-token \
  --model-dir "$GEMMA_DIR" \
  --batch-size 32 \
  --max-eval-examples 500

## Evaluate MFR + Gemma Hybrid

This is the safer scoring path: MFR handles memorized replacements, Gemma only proposes overrides for unknown candidate tokens, and predictions are accepted only when they appear in the training normalized vocabulary for that language.

Expected output:

```text
gemma candidates=...
predicting tokens=... batch_size=... batches=...
gemma predicted=... accepted=...
Baseline acc.(LAI): ...
Accuracy:           ...
ERR:                ...
wrote metrics_path=...
```

Success condition: hybrid `ERR` must be greater than the MFR baseline ERR printed above.


In [ ]:
!python mfr_gemma_hf.py \
  --eval-only \
  --use-auth-token \
  --model-dir "$GEMMA_DIR" \
  --batch-size 32 \
  --max-eval-examples 500 \
  --metrics-path outputs/gemma_hybrid_validation_subset.json \
  --predictions-path outputs/gemma_hybrid_validation_subset_predictions.json

## Full Validation Run

Run this only after the subset result is promising. It can take a while because candidate tokens still require generation.


In [ ]:
!python mfr_gemma_hf.py \
  --eval-only \
  --use-auth-token \
  --model-dir "$GEMMA_DIR" \
  --batch-size 64 \
  --metrics-path outputs/gemma_hybrid_validation_full.json \
  --predictions-path outputs/gemma_hybrid_validation_full_predictions.json

## Final Test Prediction Only

Use this after validation is good. This does not compute ERR because the test split is for prediction/submission behavior.


In [ ]:
!python mfr_gemma_hf.py \
  --predict-test \
  --use-auth-token \
  --model-dir "$GEMMA_DIR" \
  --batch-size 64 \
  --output-dir outputs/submission_mfr_gemma_hf